In [92]:
import pandas as pd
import os
import numpy as np
import re

In [93]:
pd.set_option('display.max_columns', None)

# <span style="color:blue;">**Cancer**</span>

In [94]:
study = "Cancer"

## **STEP 0: Data preparation**

### 1. DICOM

#### Visit [parameter exlanation][peid] for more information

[peid]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/Doc.aspx?sourcedoc=%7B7B05A758-0D43-40AF-9177-E9C7522D2649%7D&file=Notes_parameters%20explaination.docx&action=default&mobileredirect=true 


In [95]:
file_path = "P:/Dataset/R01-MO-DBT/MO-DBT-data-curation/dicom_tag_v2.xlsx"
dicom = pd.read_excel(file_path)

In [96]:
dicom.rename(columns={'PatientID': 'PATIENT_STUDY_ID', 'AccessionNumber': 'ACCESSION_NUMBER'}, inplace=True)

In [97]:
dicom["PATIENT_STUDY_ID"].unique().size

5577

In [98]:
PIDs = dicom["PATIENT_STUDY_ID"].unique()

In [99]:
dicom.head(5)

,PATIENT_STUDY_ID,PatientBirthDate,PatientAge,ACCESSION_NUMBER,StudyDate,Study,Side,Series,View,StudyDescription,SeriesDescription,SeriesNumber,FolderPath
0,4330018595,1965-07-01,54.0,63737104,2019-08-19,DIAG,NaN,SECURE,NaN,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,Hologic R2 ImageChecker CAD SC,1,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
1,4330018595,1965-07-01,54.0,63737104,2019-08-19,DIAG,R,FFDM,ML,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,R ML,71100000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
2,4330018595,1965-07-01,54.0,63737104,2019-08-19,DIAG,R,FFDM,XCCL,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,R XCCL,71100000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
3,4330018595,1965-07-01,54.0,63737104,2019-08-19,DIAG,L,C VIEW,LM,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,L LM C-View,71300000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
4,4330018595,1965-07-01,54.0,63737104,2019-08-19,DIAG,L,C VIEW,MLO,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,L MLO C-View,71300000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...


#### Create a list of unique study visit to serve as the reference linking the EHR to the images available for specific patient visits (as images are limited to certain visits, not all).

#### See shared parameters in [parameter exlanation][peid] to link data

[peid]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/Doc.aspx?sourcedoc=%7B7B05A758-0D43-40AF-9177-E9C7522D2649%7D&file=Notes_parameters%20explaination.docx&action=default&mobileredirect=true 

In [100]:
# Define the key identifier columns and columns to extract
key_columns = ['PATIENT_STUDY_ID', 'PatientAge', 'ACCESSION_NUMBER', 'StudyDate', 'Study', 'Side']
unique_study = dicom.drop_duplicates(subset=key_columns, keep='first')

unique_study = unique_study[key_columns]
unique_study = unique_study[unique_study['Side'].notna()]

unique_study.reset_index(drop=True, inplace=True)

In [101]:
unique_study.head(5)

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,Side
0,4330018595,54.0,63737104,2019-08-19,DIAG,R
1,4330018595,54.0,63737104,2019-08-19,DIAG,L
2,4330018595,54.0,60103700,2020-06-02,DIAG,R
3,4330018595,54.0,60690108,2020-06-02,SCREEN,L
4,4330029102,44.0,64888584,2019-05-16,SCREEN,L


In [102]:
unique_study["PATIENT_STUDY_ID"].nunique()

5572

#### Filter study with DBT 

In [103]:
dicom_dbt = dicom[dicom["Series"]=="DBT"]

key_columns = ['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'StudyDate', 'Study', 'Side']
unique_study_dicom_dbt = dicom_dbt.drop_duplicates(subset=key_columns, keep='first')
unique_study_dicom_dbt.reset_index(drop=True, inplace=True)

In [104]:
unique_study_dicom_dbt.head(5)

,PATIENT_STUDY_ID,PatientBirthDate,PatientAge,ACCESSION_NUMBER,StudyDate,Study,Side,Series,View,StudyDescription,SeriesDescription,SeriesNumber,FolderPath
0,4330066079,1989-07-01,30.0,61259016,2020-01-14,DIAG,L,DBT,CC,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,L CC Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
1,4330066079,1989-07-01,30.0,61259016,2020-01-14,DIAG,R,DBT,CC,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,R CC Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
2,4330116791,1963-07-01,56.0,61499674,2019-12-17,DIAG,L,DBT,CC,DIAG DIG MAMMO LEFT ALL VIEWS WITH TOMOSYNTHESIS,L CC Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
3,4330313855,1980-07-01,37.0,77236131,2018-05-30,DIAG,L,DBT,CC,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,L CC Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...
4,4330313855,1980-07-01,37.0,77236131,2018-05-30,DIAG,R,DBT,CC,DIAG DIG MAMMO BILAT ALL VIEWS WITH TOMOSYNTHESIS,R CC Breast Tomosynthesis Image,73200000,J:\Testing\jlee\MO_DBT\DICOM_data_and_summarie...


In [105]:
unique_study_dicom_dbt["PATIENT_STUDY_ID"].nunique()

1117

In [106]:
unique_patient_dbt = unique_study[unique_study["PATIENT_STUDY_ID"].isin(unique_study_dicom_dbt["PATIENT_STUDY_ID"].unique())]

In [107]:
unique_patient_dbt # unique patient that has dbt across stidues

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,Side
8,4330066079,30.0,61259016,2020-01-14,DIAG,L
9,4330066079,30.0,61259016,2020-01-14,DIAG,R
15,4330116791,56.0,62335422,2019-09-24,DIAG,L
16,4330116791,56.0,61499674,2019-12-17,DIAG,L
77,4330313855,37.0,77236131,2018-05-30,DIAG,L
...,...,...,...,...,...,...
26922,4339601084,46.0,77038224,2018-11-07,DIAG,L
26923,4339601084,46.0,77038224,2018-11-07,DIAG,R
26924,4339601084,46.0,65758752,2019-05-16,DIAG,R
26925,4339601084,47.0,60049293,2020-04-09,SCREEN,L


In [108]:
unique_patient_dbt = pd.merge(
    unique_patient_dbt,
    unique_study_dicom_dbt[['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'StudyDate', 'Study', 'Side', 'Series']],
    on = key_columns,
    how='left'
)

In [109]:
unique_patient_dbt

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,Side,Series
0,4330066079,30.0,61259016,2020-01-14,DIAG,L,DBT
1,4330066079,30.0,61259016,2020-01-14,DIAG,R,DBT
2,4330116791,56.0,62335422,2019-09-24,DIAG,L,NaN
3,4330116791,56.0,61499674,2019-12-17,DIAG,L,DBT
4,4330313855,37.0,77236131,2018-05-30,DIAG,L,DBT
...,...,...,...,...,...,...,...
6623,4339601084,46.0,77038224,2018-11-07,DIAG,L,DBT
6624,4339601084,46.0,77038224,2018-11-07,DIAG,R,DBT
6625,4339601084,46.0,65758752,2019-05-16,DIAG,R,DBT
6626,4339601084,47.0,60049293,2020-04-09,SCREEN,L,NaN


### 2. Electric Health Record

#### Visit [parameter exlanation][peid] for more information

[peid]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/Doc.aspx?sourcedoc=%7B7B05A758-0D43-40AF-9177-E9C7522D2649%7D&file=Notes_parameters%20explaination.docx&action=default&mobileredirect=true 


In [110]:
file_path = "P:/Dataset/R01-MO-DBT/MO-DBT-data-curation/parameters of interest.xlsx"
file_name = pd.ExcelFile(file_path).sheet_names
file_name

['enteredit_findings',
 'pathology',
 'pathology_findings',
 'patient_data_ie',
 'hormonal_mens',
 'risk_factors',
 'vitals',
 'patient_demo',
 'procedure_notes']

---

# **Extract <span style="color:blue;">MO Cancer**</span> 

#### See [inclusion criteria][ic] to for filtering

[ic]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/doc.aspx?sourcedoc=%7B9c403413-a771-4830-98ab-15171872c4dd%7D&action=edit

## **STEP 1**. Merge enteredit_findings with pathology, then merge with patient with DBT available
### (OUTPUT) cancer_cohort

enteredit_findings
* Columns: COMPOSITION_NAME, FINDING_LOCATION, FINDING_CATEGORY, FINDING_REC, EXAM_COMPLETED_DATE
* Cancer: 
    * FINDING_CATEGORY = 4, 4A, 4B, 4C, 5

pathology
* Columns: BX_ID, PATHOLOGY_DATE, LESION_CLASS, SIDE
* Cancer: 
    * LESION_CLASS = 'Malignant'

In [111]:
file_name

['enteredit_findings',
 'pathology',
 'pathology_findings',
 'patient_data_ie',
 'hormonal_mens',
 'risk_factors',
 'vitals',
 'patient_demo',
 'procedure_notes']

In [112]:
fn = file_name[0]
file_path = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, "Cleaned", fn + ".xlsx")
birads = pd.read_excel(file_path)

In [113]:
birads.head(3)

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count
0,4330018595,63027507,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2019-07-26,2
1,4330018595,63737104,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2019-08-19,1
2,4330018595,63737104,Heterogeneously dense (51% - 75%),NaN,3 - Probably benign - short interval follow-up,F-Follow-up at short interval (1-11 months),2019-08-19,1


In [114]:
fn = file_name[1]
file_path = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, "Cleaned", fn + ".xlsx")
pathology = pd.read_excel(file_path)

In [115]:
pathology.head(3)

,PATIENT_STUDY_ID,BX_ID,PATHOLOGY_DATE,LESION_CLASS,SIDE
0,4330018595,487690,2020-06-05,Malignant,R
1,4330018595,475789,2020-07-28,Malignant,R
2,4330029102,477464,2021-05-10,Benign,R


In [116]:
biopsy_categories = [
    '4 - Suspicious abnormality, biopsy should be considered',
    '4A - Suspicious abnormality - biopsy should be considered - low suspicion',
    '4B - Suspicious abnormality - biopsy should be considered - intermediate suspicion',
    '4C - Suspicious abnormality - biopsy should be considered - moderate suspicion',
    '5 - Highly suggestive of malignancy, appropriate action should be taken'
]

In [117]:
# ── STEP 1: Merge birads + pathology ────────────────────────────────────────

birads['EXAM_COMPLETED_DATE'] = pd.to_datetime(birads['EXAM_COMPLETED_DATE'])
pathology['PATHOLOGY_DATE']   = pd.to_datetime(pathology['PATHOLOGY_DATE'])
birads['needs_biopsy']        = birads['FINDING_CATEGORY'].isin(biopsy_categories)

# STEP 1.1: Biopsy findings → link to pathology via date window
birads_biopsy    = birads[birads['needs_biopsy']].copy()
biopsy_with_path = pd.merge(birads_biopsy, pathology, on='PATIENT_STUDY_ID', how='left')
biopsy_with_path = biopsy_with_path[
    (biopsy_with_path['PATHOLOGY_DATE'] >= biopsy_with_path['EXAM_COMPLETED_DATE']) &
    (biopsy_with_path['PATHOLOGY_DATE'] <= biopsy_with_path['EXAM_COMPLETED_DATE'] + pd.DateOffset(months=6))
]
biopsy_with_path['days_diff'] = (
    biopsy_with_path['PATHOLOGY_DATE'] - biopsy_with_path['EXAM_COMPLETED_DATE']
).dt.days
biopsy_with_path = (
    biopsy_with_path
    .sort_values('days_diff')
    .drop_duplicates(subset=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'SIDE'], keep='first')
    .drop(columns='days_diff')
)

# STEP 1.2: Non-biopsy findings → assign opposite side,
#           but ONLY when exactly one biopsy side exists (L+R = ambiguous → skip)
birads_no_biopsy = birads[~birads['needs_biopsy']].copy()

biopsy_sides = (
    biopsy_with_path[['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'SIDE']]
    .drop_duplicates()
    .rename(columns={'SIDE': 'biopsy_side'})
)
single_side_accessions = (
    biopsy_sides
    .groupby(['PATIENT_STUDY_ID', 'ACCESSION_NUMBER'])
    .filter(lambda g: len(g) == 1)
)

birads_no_biopsy = pd.merge(
    birads_no_biopsy, single_side_accessions,
    on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER'], how='left'
)
birads_no_biopsy['SIDE'] = birads_no_biopsy['biopsy_side'].map({'L': 'R', 'R': 'L'})
birads_no_biopsy = birads_no_biopsy.drop(columns='biopsy_side')

# STEP 1.3: Rows still without SIDE, accession has NO biopsy at all,
#           and duplicate_count >= 2 → bilateral exam → duplicate as L and R
biopsy_accession_keys = biopsy_sides[['PATIENT_STUDY_ID', 'ACCESSION_NUMBER']].drop_duplicates()

no_side_rows  = birads_no_biopsy[birads_no_biopsy['SIDE'].isna()].copy()
has_side_rows = birads_no_biopsy[birads_no_biopsy['SIDE'].notna()].copy()

no_side_rows = no_side_rows.merge(
    biopsy_accession_keys, on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER'],
    how='left', indicator=True
)
truly_no_biopsy  = no_side_rows['_merge'] == 'left_only'
no_side_rows     = no_side_rows.drop(columns='_merge')

bilateral_mask = truly_no_biopsy & (no_side_rows['duplicate_count'] >= 2)
bilateral      = no_side_rows[bilateral_mask]
rest           = no_side_rows[~bilateral_mask]

bilateral_L = bilateral.copy(); bilateral_L['SIDE'] = 'L'
bilateral_R = bilateral.copy(); bilateral_R['SIDE'] = 'R'

birads_no_biopsy = pd.concat(
    [has_side_rows, bilateral_L, bilateral_R, rest], ignore_index=True
)

birads_with_path = pd.concat([biopsy_with_path, birads_no_biopsy], ignore_index=True)

In [118]:
# ── STEP 2: Connect to DICOM images (unique_patient_dbt) ─────────────────────

unique_patient_dbt = unique_patient_dbt.rename(columns={'Side': 'SIDE'})

has_side = (birads_with_path[birads_with_path['SIDE'].notna()]
            .drop_duplicates(subset=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'SIDE'], keep='first'))
no_side  = (birads_with_path[birads_with_path['SIDE'].isna()]
            .drop_duplicates(subset=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER'], keep='first')
            .drop(columns='SIDE'))

# 1. Sided match
merge_sided = pd.merge(
    unique_patient_dbt, has_side,
    on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'SIDE'], how='inner'
)

# 2. No-side match — only DICOM rows whose accession wasn't already matched above
sided_accessions = merge_sided[['PATIENT_STUDY_ID', 'ACCESSION_NUMBER']].drop_duplicates()
dicom_unmatched  = unique_patient_dbt.merge(
    sided_accessions, on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER'],
    how='left', indicator=True
).query('_merge == "left_only"').drop(columns='_merge')

merge_no_side = pd.merge(
    dicom_unmatched, no_side,
    on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER'], how='inner'
)

# 3. DICOM rows with no birads match at all → keep with NaN birads columns
matched_keys = pd.concat([
    merge_sided[['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'SIDE']],
    merge_no_side[['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'SIDE']]
]).drop_duplicates()

no_birads = unique_patient_dbt.merge(
    matched_keys, on=['PATIENT_STUDY_ID', 'ACCESSION_NUMBER', 'SIDE'],
    how='left', indicator=True
).query('_merge == "left_only"').drop(columns='_merge')

df_final = pd.concat([merge_sided, merge_no_side, no_birads], ignore_index=True)
df_final = df_final.sort_values(['PATIENT_STUDY_ID', 'StudyDate', 'ACCESSION_NUMBER', 'SIDE'], ignore_index=True).drop(columns=['duplicate_count', 'needs_biopsy'])

print(f"unique_patient_dbt : {unique_patient_dbt.shape[0]} rows")
print(f"  ↳ with side      : {len(merge_sided)}")
print(f"  ↳ without side   : {len(merge_no_side)}")
print(f"  ↳ no birads      : {len(no_birads)}")
print(f"df_final           : {df_final.shape[0]} rows")

unique_patient_dbt : 6628 rows
  ↳ with side      : 4754
  ↳ without side   : 1677
  ↳ no birads      : 197
df_final           : 6628 rows


In [119]:
df_final.loc[df_final['FINDING_CATEGORY'].isna(), 'PATIENT_STUDY_ID'].nunique()

126

In [120]:
unique_patient_dbt[unique_patient_dbt['PATIENT_STUDY_ID']==4339601084]

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,SIDE,Series
6623,4339601084,46.0,77038224,2018-11-07,DIAG,L,DBT
6624,4339601084,46.0,77038224,2018-11-07,DIAG,R,DBT
6625,4339601084,46.0,65758752,2019-05-16,DIAG,R,DBT
6626,4339601084,47.0,60049293,2020-04-09,SCREEN,L,NaN
6627,4339601084,47.0,60049293,2020-04-09,SCREEN,R,NaN


In [121]:
df_final[df_final['PATIENT_STUDY_ID']==4339601084]

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,SIDE,Series,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,BX_ID,PATHOLOGY_DATE,LESION_CLASS
6623,4339601084,46.0,77038224,2018-11-07,DIAG,L,DBT,NaN,NaN,NaN,NaN,NaT,NaN,NaT,NaN
6624,4339601084,46.0,77038224,2018-11-07,DIAG,R,DBT,Heterogeneously dense (51% - 75%),NaN,"4 - Suspicious abnormality, biopsy should be c...",B-Biopsy should be considered,2018-11-07,427183.0,2018-11-14,Benign
6625,4339601084,46.0,65758752,2019-05-16,DIAG,R,DBT,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2019-05-16,NaN,NaT,NaN
6626,4339601084,47.0,60049293,2020-04-09,SCREEN,L,NaN,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2020-04-09,NaN,NaT,NaN
6627,4339601084,47.0,60049293,2020-04-09,SCREEN,R,NaN,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2020-04-09,NaN,NaT,NaN


In [122]:
birads[birads['PATIENT_STUDY_ID']==4339601084]

,PATIENT_STUDY_ID,ACCESSION_NUMBER,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,duplicate_count,needs_biopsy
87049,4339601084,70648957,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2017-10-11,2,False
87050,4339601084,79474989,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,U-Ultrasound,2017-11-07,1,False
87051,4339601084,79474989,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2017-11-07,1,False
87052,4339601084,79253799,Heterogeneously dense (51% - 75%),NaN,"4 - Suspicious abnormality, biopsy should be c...",B-Biopsy should be considered,2017-11-16,1,True
87053,4339601084,78102132,Heterogeneously dense (51% - 75%),NaN,3 - Probably benign - short interval follow-up,F-Follow-up at short interval (1-11 months),2018-05-24,1,False
87054,4339601084,77038224,Heterogeneously dense (51% - 75%),NaN,"4 - Suspicious abnormality, biopsy should be c...",B-Biopsy should be considered,2018-11-07,1,True
87055,4339601084,65758752,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2019-05-16,1,False
87056,4339601084,60049293,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2020-04-09,2,False
87057,4339601084,67077961,Scattered fibroglandular (25% - 50%),NaN,2 - Benign finding,N-Normal interval follow-up,2021-04-15,2,False
87058,4339601084,452833482,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2022-04-25,2,False


In [123]:
pathology[pathology['PATIENT_STUDY_ID']==4339601084]

,PATIENT_STUDY_ID,BX_ID,PATHOLOGY_DATE,LESION_CLASS,SIDE
11297,4339601084,436123,2017-11-22,Benign,R
11298,4339601084,436122,2017-11-22,Benign,R
11299,4339601084,436121,2017-11-22,Benign,R
11300,4339601084,427183,2018-11-14,Benign,R


In [124]:
df_step1 = df_final.copy()

In [125]:
df_step1.shape

(6628, 15)

In [126]:
df_step1["EXAM_COMPLETED_DATE"] = df_step1["EXAM_COMPLETED_DATE"].dt.strftime("%Y-%m-%d")
df_step1["PATHOLOGY_DATE"] = df_step1["PATHOLOGY_DATE"].dt.strftime("%Y-%m-%d")

### <span style="color:#FF6347;">**SAVE**</span> file

In [127]:
output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'cancer_cohort' + ".xlsx")
df_step1.to_excel(output_file, index=False)

## **STEP 2**. <span style="color:blue;">**MO cancer**</span>: Label  "Index" = <span style="color:#8A2BE2;">**INDEX**</span> or <span style="color:#00BFFF;">**INDEX-1**</span>
### (OUTPUT) cancer_cohort_index

### <span style="color:#FF6347;">**READ**</span> file (cancer_cohort)

In [128]:
output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'cancer_cohort' + ".xlsx")
cancer_cohort = pd.read_excel(output_file)

In [129]:
cancer_cohort["EXAM_COMPLETED_DATE"] = pd.to_datetime(cancer_cohort["EXAM_COMPLETED_DATE"], format="%Y-%m-%d")
cancer_cohort["PATHOLOGY_DATE"] = pd.to_datetime(cancer_cohort["PATHOLOGY_DATE"], format="%Y-%m-%d")

In [130]:
cancer_cohort["PATIENT_STUDY_ID"].nunique()

1117

In [131]:
cancer_cohort.head(5)

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,SIDE,Series,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,BX_ID,PATHOLOGY_DATE,LESION_CLASS
0,4330066079,30.0,61259016,2020-01-14,DIAG,L,DBT,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2020-01-14,NaN,NaT,NaN
1,4330066079,30.0,61259016,2020-01-14,DIAG,R,DBT,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2020-01-14,NaN,NaT,NaN
2,4330116791,56.0,62335422,2019-09-24,DIAG,L,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaT,NaN
3,4330116791,56.0,61499674,2019-12-17,DIAG,L,DBT,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2019-12-17,NaN,NaT,NaN
4,4330313855,37.0,77236131,2018-05-30,DIAG,L,DBT,Extremely dense (>75%),NaN,4B - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2018-05-30,415556.0,2018-06-05,Benign


## 1. Locate <span style="color:blue;">**Cancer**</span> year, "Index" = <span style="color:#8A2BE2;">**INDEX**</span>

#### See [inclusion criteria][ic] to for filtering

[ic]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/doc.aspx?sourcedoc=%7B9c403413-a771-4830-98ab-15171872c4dd%7D&action=edit

In [132]:
# ── Find earliest Malignant pathology per patient → INDEX date ───────────────

cohort = cancer_cohort.copy()
cohort['Index'] = None

# 1. Find the earliest Malignant PATHOLOGY_DATE per patient
first_malignant = (
    cohort[cohort['LESION_CLASS'] == 'Malignant']
    .sort_values('PATHOLOGY_DATE')
    .drop_duplicates(subset=['PATIENT_STUDY_ID'], keep='first')
    [['PATIENT_STUDY_ID', 'PATHOLOGY_DATE']]
    .rename(columns={'PATHOLOGY_DATE': 'target pathology date'})
)

cohort = cohort.merge(first_malignant, on='PATIENT_STUDY_ID', how='left')

# 2. Mark exams within 6 months BEFORE (or on) the index pathology date
six_months = pd.Timedelta(days=365 / 12 * 6)

cohort['path to exam diff'] = cohort['target pathology date'] - cohort['EXAM_COMPLETED_DATE']

mask_index_cohort = (
    (cohort['path to exam diff'] >= pd.Timedelta(days=0)) &
    (cohort['path to exam diff'] <= six_months)
)

cohort.loc[mask_index_cohort, 'Index'] = 'INDEX'

print(f"✅ Marked {cohort['Index'].eq('INDEX').sum()} rows as 'INDEX'.")

✅ Marked 395 rows as 'INDEX'.


In [133]:
cohort.reset_index(drop=True, inplace=True)

In [134]:
cohort

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,SIDE,Series,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,BX_ID,PATHOLOGY_DATE,LESION_CLASS,Index,target pathology date,path to exam diff
0,4330066079,30.0,61259016,2020-01-14,DIAG,L,DBT,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2020-01-14,NaN,NaT,NaN,None,NaT,NaT
1,4330066079,30.0,61259016,2020-01-14,DIAG,R,DBT,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2020-01-14,NaN,NaT,NaN,None,NaT,NaT
2,4330116791,56.0,62335422,2019-09-24,DIAG,L,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaT,NaN,None,NaT,NaT
3,4330116791,56.0,61499674,2019-12-17,DIAG,L,DBT,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2019-12-17,NaN,NaT,NaN,None,NaT,NaT
4,4330313855,37.0,77236131,2018-05-30,DIAG,L,DBT,Extremely dense (>75%),NaN,4B - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2018-05-30,415556.0,2018-06-05,Benign,None,NaT,NaT
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6623,4339601084,46.0,77038224,2018-11-07,DIAG,L,DBT,NaN,NaN,NaN,NaN,NaT,NaN,NaT,NaN,None,NaT,NaT
6624,4339601084,46.0,77038224,2018-11-07,DIAG,R,DBT,Heterogeneously dense (51% - 75%),NaN,"4 - Suspicious abnormality, biopsy should be c...",B-Biopsy should be considered,2018-11-07,427183.0,2018-11-14,Benign,None,NaT,NaT
6625,4339601084,46.0,65758752,2019-05-16,DIAG,R,DBT,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2019-05-16,NaN,NaT,NaN,None,NaT,NaT
6626,4339601084,47.0,60049293,2020-04-09,SCREEN,L,NaN,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2020-04-09,NaN,NaT,NaN,None,NaT,NaT


## 2. Locate <span style="color:blue;">**MO cancer**</span> year, "Index" = <span style="color:#00BFFF;">**INDEX-1**</span>

#### See [inclusion criteria][ic] to for filtering

[ic]: https://pitt-my.sharepoint.com/:w:/r/personal/trl117_pitt_edu/_layouts/15/doc.aspx?sourcedoc=%7B9c403413-a771-4830-98ab-15171872c4dd%7D&action=edit

In [135]:
# ── Find INDEX-1: exam between 9–18 months before earliest INDEX exam ─────────

nine_months     = pd.Timedelta(days=365 / 12 * 9)
eighteen_months = pd.Timedelta(days=365 / 12 * 18)

# 1. Get the earliest INDEX exam date per patient
earliest_index = (
    cohort[cohort['Index'] == 'INDEX']
    .groupby('PATIENT_STUDY_ID')['EXAM_COMPLETED_DATE']
    .min()
    .reset_index()
    .rename(columns={'EXAM_COMPLETED_DATE': 'earliest index date'})
)

cohort = cohort.merge(earliest_index, on='PATIENT_STUDY_ID', how='left')

# 2. Time between each exam and the earliest INDEX exam
cohort['exam to index diff'] = cohort['earliest index date'] - cohort['EXAM_COMPLETED_DATE']

# 3. Window: at least 9 months before, no more than 18 months before
mask_window = (
    (cohort['exam to index diff'] >= nine_months) &
    (cohort['exam to index diff'] <= eighteen_months)
)

cohort.loc[mask_window, 'Index'] = 'INDEX-1'

# # 4. Among candidates, keep the one closest to INDEX (largest exam date = smallest diff)
# index_minus_1_idx = (
#     cohort[mask_window]
#     .sort_values('exam to index diff')                          # smallest diff first
#     .drop_duplicates(subset=['PATIENT_STUDY_ID'], keep='first') # closest per patient
#     .index
# )

# cohort.loc[index_minus_1_idx, 'Index'] = 'INDEX-1'

print(f"✅ Marked {cohort['Index'].eq('INDEX-1').sum()} rows as 'INDEX-1'.")
print(f"   Patients with INDEX-1 : {cohort[cohort['Index'] == 'INDEX-1']['PATIENT_STUDY_ID'].nunique()}")
print(f"   Patients with INDEX   : {cohort[cohort['Index'] == 'INDEX']['PATIENT_STUDY_ID'].nunique()}")

✅ Marked 92 rows as 'INDEX-1'.
   Patients with INDEX-1 : 47
   Patients with INDEX   : 200


In [136]:
cohort[cohort['Index'] == 'INDEX-1']['PATIENT_STUDY_ID'].unique()

array([4333010687, 4333012288, 4333012936, 4333023156, 4333023918,
       4333041185, 4333062996, 4333063695, 4333076307, 4333081537,
       4333090748, 4333093019, 4333097425, 4333326250, 4333335215,
       4333349729, 4333367699, 4333376924, 4333409947, 4333416464,
       4333421883, 4333473906, 4333505027, 4333587108, 4333603027,
       4333603038, 4333604775, 4333608272, 4333609483, 4333663896,
       4333666401, 4333957642, 4333969805, 4334005756, 4334046824,
       4334051051, 4334054936, 4334062099, 4334064465, 4334096599,
       4334345819, 4334510859, 4334511244, 4334582108, 4334652434,
       4334965484, 4335808308], dtype=int64)

In [137]:
cohort[cohort['PATIENT_STUDY_ID'] == 4333010687]

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,SIDE,Series,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,BX_ID,PATHOLOGY_DATE,LESION_CLASS,Index,target pathology date,path to exam diff,earliest index date,exam to index diff
174,4333010687,64.0,77464018,2018-07-07,SCREEN,L,NaN,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2018-07-07,NaN,NaT,NaN,INDEX-1,2020-02-17,590 days,2019-12-21,532 days
175,4333010687,64.0,77464018,2018-07-07,SCREEN,R,NaN,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2018-07-07,NaN,NaT,NaN,INDEX-1,2020-02-17,590 days,2019-12-21,532 days
176,4333010687,65.0,62010914,2019-12-21,SCREEN,L,NaN,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2019-12-21,NaN,NaT,NaN,INDEX,2020-02-17,58 days,2019-12-21,0 days
177,4333010687,65.0,62010914,2019-12-21,SCREEN,R,NaN,Heterogeneously dense (51% - 75%),NaN,0 - Need additional imaging evaluation,P-Additional projections,2019-12-21,NaN,NaT,NaN,INDEX,2020-02-17,58 days,2019-12-21,0 days
178,4333010687,65.0,61024786,2020-01-17,DIAG,L,NaN,Heterogeneously dense (51% - 75%),NaN,4B - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2020-01-17,481678.0,2020-02-17,Malignant,INDEX,2020-02-17,31 days,2019-12-21,-27 days
179,4333010687,67.0,453985357,2022-04-06,SCREEN,R,DBT,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2022-04-06,NaN,NaT,NaN,None,2020-02-17,-779 days,2019-12-21,-837 days


In [138]:
cohort = cohort.drop(columns = ["target pathology date", "path to exam diff", "earliest index date", "exam to index diff"])

In [139]:
cohort.head(5)

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,SIDE,Series,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,BX_ID,PATHOLOGY_DATE,LESION_CLASS,Index
0,4330066079,30.0,61259016,2020-01-14,DIAG,L,DBT,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2020-01-14,NaN,NaT,NaN,None
1,4330066079,30.0,61259016,2020-01-14,DIAG,R,DBT,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2020-01-14,NaN,NaT,NaN,None
2,4330116791,56.0,62335422,2019-09-24,DIAG,L,NaN,NaN,NaN,NaN,NaN,NaT,NaN,NaT,NaN,None
3,4330116791,56.0,61499674,2019-12-17,DIAG,L,DBT,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2019-12-17,NaN,NaT,NaN,None
4,4330313855,37.0,77236131,2018-05-30,DIAG,L,DBT,Extremely dense (>75%),NaN,4B - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2018-05-30,415556.0,2018-06-05,Benign,None


### <span style="color:#FF6347;">**SAVE**</span> file (cancer_cohort_index)

In [140]:
cohort["EXAM_COMPLETED_DATE"] = cohort["EXAM_COMPLETED_DATE"].dt.strftime("%Y-%m-%d")
cohort["PATHOLOGY_DATE"] = cohort["PATHOLOGY_DATE"].dt.strftime("%Y-%m-%d")

In [141]:
output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'cancer_cohort_index' + ".xlsx")
cohort.to_excel(output_file, index=False)

## **STEP 3**. Extract <span style="color:blue;">**MO cancer**</span> cohort only
### (Output) mo_cancer, mo_cancer_cohort

#### <span style="color:#FF6347;">**READ**</span> file (cancer_cohort)

In [142]:
output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'cancer_cohort_index' + ".xlsx")
cohort = pd.read_excel(output_file)

In [143]:
cohort["EXAM_COMPLETED_DATE"] = pd.to_datetime(cohort["EXAM_COMPLETED_DATE"], format="%Y-%m-%d")
cohort["PATHOLOGY_DATE"] = pd.to_datetime(cohort["PATHOLOGY_DATE"], format="%Y-%m-%d")

### **Criteria**:
#### 1. At <span style="color:#8A2BE2;">**INDEX**</span>, "LESION_CLASS" = "Malignant"
##### no need to check as all patients in cancer_cohort already meet this criteria
#### 2. At <span style="color:#00BFFF;">**INDEX-1**</span>, "Series" = "DBT"
#### 3. At <span style="color:#00BFFF;">**INDEX-1**</span>, "FINDING_CATEGORY" = "1 - Negative", "2 - Benign finding"
#### 4. At <span style="color:#00BFFF;">**INDEX-1**</span>, "Study" = "SCREEN"

In [144]:
idx_index_1 = cohort["Index"]=="INDEX-1"
idx_dbt = cohort["Series"]=="DBT"
idx_screen = cohort["Study"]=="SCREEN"
idx_birads_0 = cohort["FINDING_CATEGORY"]=="0 - Need additional imaging evaluation"
idx_birads_1 = cohort["FINDING_CATEGORY"]=="1 - Negative"
idx_birads_2 = cohort["FINDING_CATEGORY"]=="2 - Benign finding"

#### 3.1. [N=14] Meet **criteria #2**

In [145]:
pids_criteria_2 = cohort[idx_index_1 & idx_dbt]["PATIENT_STUDY_ID"].unique()

In [146]:
len(pids_criteria_2)

14

#### 3.2. [N=13] Meet **criteria #2 <span style="color:red">**AND**</span> #3**

In [147]:
pids_criteria_2_3 = cohort[(idx_index_1 & idx_dbt & idx_birads_1) | (idx_index_1 & idx_dbt & idx_birads_2)]["PATIENT_STUDY_ID"].unique()

In [148]:
len(pids_criteria_2_3)

13

#### 3.3. [N=208] Meet **criteria #2 <span style="color:red">**AND**</span> #3 <span style="color:red">**AND**</span> #4**  → Remove PID with other BIRADS score (not 1 or 2) at <span style="color:#00BFFF;">**INDEX-1**</span> → <span style="color:#FF6347;">**SAVE AS**</span> **mo_cancer_cohort**

##### 3.3.A. Meet **criteria #2 <span style="color:red">**AND**</span> #3 <span style="color:red">**AND**</span> #4**

In [149]:
pids_criteria_all = cohort[(idx_screen & idx_index_1 & idx_dbt & idx_birads_1) | (idx_screen & idx_index_1 & idx_dbt & idx_birads_2)]["PATIENT_STUDY_ID"].unique()

In [150]:
len(pids_criteria_all)

7

##### Exclude PIDs with BIRADS that is not 1 or 2

In [151]:
findings_to_exclude = ["1 - Negative", "2 - Benign finding"]

##### 3.3.B. Remove PID with any BIRADS score other than 1 or 2 at <span style="color:#00BFFF;">**INDEX-1**</span> 

In [152]:
tmp_cohort_index_1 = cohort[idx_index_1]

In [153]:
filter_mask = ~tmp_cohort_index_1['FINDING_CATEGORY'].isin(findings_to_exclude)
pids_exclude_index_1 = tmp_cohort_index_1[filter_mask]['PATIENT_STUDY_ID'].tolist()

##### $A - B$

In [154]:
set_A = set(pids_criteria_all)
set_B = set(pids_exclude_index_1)

In [155]:
final_pids = list(set_A.difference(set_B))

In [156]:
len(final_pids)

7

##### <span style="color:#FF6347;">**SAVE**</span> **mo_cancer_cohort** (full record we have for MO patients with images)

In [157]:
cohort_filtered = cohort[cohort['PATIENT_STUDY_ID'].isin(final_pids)].reset_index(drop=True)

In [158]:
cohort_filtered_sort = cohort_filtered.sort_values(['PATIENT_STUDY_ID', 'StudyDate', 'ACCESSION_NUMBER']).reset_index(drop=True)

In [159]:
cohort_filtered_sort[cohort_filtered_sort["PATIENT_STUDY_ID"]==final_pids[0]]

,PATIENT_STUDY_ID,PatientAge,ACCESSION_NUMBER,StudyDate,Study,SIDE,Series,COMPOSITION_NAME,FINDING_LOCATION,FINDING_CATEGORY,FINDING_REC,EXAM_COMPLETED_DATE,BX_ID,PATHOLOGY_DATE,LESION_CLASS,Index
0,4333041185,45.0,72905651,2017-03-27,DIAG,L,NaN,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2017-03-27,NaN,NaT,NaN,NaN
1,4333041185,45.0,72905651,2017-03-27,DIAG,R,NaN,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2017-03-27,NaN,NaT,NaN,NaN
2,4333041185,47.0,76820901,2018-11-26,SCREEN,L,NaN,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2018-11-26,NaN,NaT,NaN,NaN
3,4333041185,47.0,76820901,2018-11-26,SCREEN,R,NaN,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2018-11-26,NaN,NaT,NaN,NaN
4,4333041185,48.0,62018362,2019-11-01,SCREEN,L,NaN,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2019-11-01,NaN,NaT,NaN,NaN
5,4333041185,48.0,62018362,2019-11-01,SCREEN,R,NaN,Heterogeneously dense (51% - 75%),NaN,1 - Negative,N-Normal interval follow-up,2019-11-01,NaN,NaT,NaN,NaN
6,4333041185,50.0,454994176,2021-11-03,SCREEN,L,DBT,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2021-11-03,NaN,NaT,NaN,INDEX-1
7,4333041185,50.0,454994176,2021-11-03,SCREEN,R,DBT,Heterogeneously dense (51% - 75%),NaN,2 - Benign finding,N-Normal interval follow-up,2021-11-03,NaN,NaT,NaN,INDEX-1
8,4333041185,51.0,451483856,2022-08-05,DIAG,L,DBT,Heterogeneously dense (51% - 75%),NaN,4A - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2022-08-05,353483.0,2022-09-02,Malignant,INDEX
9,4333041185,51.0,451483856,2022-08-05,DIAG,R,DBT,Heterogeneously dense (51% - 75%),NaN,4A - Suspicious abnormality - biopsy should be...,B-Biopsy should be considered,2022-08-05,354429.0,2022-08-30,Benign,INDEX


In [160]:
df_step3 = cohort_filtered_sort

In [161]:
df_step3["EXAM_COMPLETED_DATE"] = df_step3["EXAM_COMPLETED_DATE"].dt.strftime("%Y-%m-%d")
df_step3["PATHOLOGY_DATE"] = df_step3["PATHOLOGY_DATE"].dt.strftime("%Y-%m-%d")

In [162]:
output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'mo_cancer_cohort' + ".xlsx")
df_step3.to_excel(output_file, index=False)

##### <span style="color:#FF6347;">**SAVE**</span> **mo_cancer** (only retain INDX & INDEX-1)

In [163]:
df_final = df_step3[(df_step3['Index']=="INDEX") | (df_step3['Index']=="INDEX-1")]

In [164]:
output_file = os.path.join("P:/Dataset/R01-MO-DBT/MO-DBT-data-curation", study, 'mo_cancer' + ".xlsx")
df_final.to_excel(output_file, index=False)